In [1]:
!pip install easyocr -q

In [2]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from collections import defaultdict

random.seed(42)

deva_base = "/kaggle/input/datasets/ashokpant/devanagari-character-dataset/nhcd/nhcd"
bangla_base = "/kaggle/input/datasets/asefjamilajwad2/banglalekha-isolated/BanglaLekha-Isolated/Images"

print("✅ Imports done")

✅ Imports done


In [3]:
def get_balanced_samples(base_path, ext, samples_per_class=50):
    class_images = defaultdict(list)
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(ext):
                class_name = os.path.basename(root)
                class_images[class_name].append(os.path.join(root, file))
    selected = []
    for class_name, images in class_images.items():
        random.shuffle(images)
        for img_path in images[:samples_per_class]:
            selected.append((img_path, class_name))
    return selected

hindi_data = get_balanced_samples(deva_base, '.jpg', 50)
bengali_data = get_balanced_samples(bangla_base, '.png', 50)
all_data = hindi_data + bengali_data

all_labels = sorted(set([label for _, label in all_data]))
label2idx = {label: idx for idx, label in enumerate(all_labels)}
num_classes = len(all_labels)

print(f"Hindi samples: {len(hindi_data)}")
print(f"Bengali samples: {len(bengali_data)}")
print(f"Total: {len(all_data)}")
print(f"Classes: {num_classes}")

Hindi samples: 1850
Bengali samples: 4200
Total: 6050
Classes: 94


In [6]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class IndicDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        return img, label2idx[label]

dataset = IndicDataset(all_data, transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {len(dataloader)}")
print("✅ Dataloader ready")

Dataset size: 6050
Batches per epoch: 190
✅ Dataloader ready


In [7]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.rnn = nn.LSTM(128 * 4, 256, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, w, c * h)
        x, _ = self.rnn(x)
        x = self.fc(x[:, -1, :])
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = CRNN(num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f} | Accuracy: {acc:.2f}%")

torch.save(model.state_dict(), '/kaggle/working/indic_crnn_finetuned.pth')
print("\n✅ Weights saved to /kaggle/working/indic_crnn_finetuned.pth")

Using: cuda
Epoch 1/10 | Loss: 3.9103 | Accuracy: 7.12%
Epoch 2/10 | Loss: 3.0189 | Accuracy: 21.45%
Epoch 3/10 | Loss: 2.4432 | Accuracy: 33.97%
Epoch 4/10 | Loss: 1.9561 | Accuracy: 46.91%
Epoch 5/10 | Loss: 1.5732 | Accuracy: 57.11%
Epoch 6/10 | Loss: 1.2325 | Accuracy: 66.98%
Epoch 7/10 | Loss: 0.9737 | Accuracy: 74.71%
Epoch 8/10 | Loss: 0.6862 | Accuracy: 82.98%
Epoch 9/10 | Loss: 0.4759 | Accuracy: 89.24%
Epoch 10/10 | Loss: 0.3202 | Accuracy: 93.70%

✅ Weights saved to /kaggle/working/indic_crnn_finetuned.pth
